## Section: Regression and scorers

#### **Exercise 1** 

Here is some sample data.  

In [ ]:
import numpy as np
# Sample data
y_true = np.array([3, -0.5, 2, 7])
y_pred = np.array([2.5, 0.0, 2, 8])

Create functions that implement R^2, RMSE, and MAE in python. Each function should have the same api (meaning it will take the same parameters), like

~~~python
def function_name(y_true,y_pred):
    # logic for your measures
    # make sure to return a result!
~~~

Name your functions `r2_score`, `rmse`, and `mae`.

In [ ]:
#Code here

Run the following cell once your functions are defined to see the result.

In [ ]:
print(f"R² Score: {r2_score(y_true, y_pred):.4f}")
print(f"rmse Score: {rmse(y_true, y_pred):.4f}")
print(f"MAE: {mae(y_true, y_pred):.4f}")

#### **Exercise 2**

In this exercise, you will define a custom scorer (using `make_scorer`) to evaluate a solution to simulated ML problem. You will compare scores for different combinations of pre-processing steps.


**ML Problem definition**

Dataset features:
1. 'area': House area in square feet (numeric)
2. 'bedrooms': Number of bedrooms (numeric)
3. 'age': Age of the house in years (numeric)
4. 'neighborhood': Categorical feature with missing values (categorical)
5. 'distance_to_city_center': Distance to city center in miles (exponential feature)

Target variable:
- 'price': House price in thousands of dollars

I have developed an ML solution in the function `process_ml` that take parameters to control it's preprocessing steps.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer

# Generate synthetic data
np.random.seed(42)
n_samples = 1000

area = np.random.uniform(1000, 5000, n_samples)
bedrooms = np.random.randint(1, 6, n_samples)
age = np.random.uniform(1, 50, n_samples)
neighborhood = np.random.choice(['A', 'B', 'C', 'D', 'E'], n_samples).astype(object)
distance_to_city_center = np.random.exponential(scale=5, size=n_samples)

# Introduce missing values in neighborhood
neighborhood[np.random.choice(n_samples, 100, replace=False)] = np.nan

# Create target variable with log-scaled distance and neighborhood effect
price = (
    10 * np.log(area) +
    5 * bedrooms -
    2 * age +
    np.where(neighborhood == 'A', 50, 0) +
    np.where(neighborhood == 'C', 30, 0) +
    np.where(neighborhood == 'E', 10, 0) +
    np.where(neighborhood == 'D', -20, 0) +
    np.where(neighborhood == 'B', -40, 0) -
    20 * np.log(distance_to_city_center + 1) +
    np.random.normal(0, 10, n_samples)
)

# Create DataFrame
df = pd.DataFrame({
    'area': area,
    'bedrooms': bedrooms,
    'age': age,
    'neighborhood': neighborhood,
    'distance_to_city_center': distance_to_city_center,
    'price_10K': price
})

X = df.drop('price_10K', axis=1)
y = df['price_10K']

# ML Processing Pipeline

def ml_pipeline(encoder="ordinal", log_scale=True):
    numeric_features = ['area', 'bedrooms', 'age','distance_to_city_center']
    
    numeric_transformer = SimpleImputer(strategy='median')
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OrdinalEncoder() if encoder == 'ordinal' else OneHotEncoder(drop='first', sparse_output=False))
    ])
    col_transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, ['neighborhood'])
        ]
    
    if log_scale:
        col_transformers.append(('log_transform', FunctionTransformer(np.log1p), ['distance_to_city_center']))

    preprocessor = ColumnTransformer(col_transformers)
        
    return Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', LinearRegression())])
    

Your task is to complete the `evaluate` function below so that it runs 5-fold cross-validation on each pipeline, scoring it with **your three functions from Exercise 1**.

1. Build `scoring_function` as a dictionary of scorers, one per metric, by wrapping each of your functions with `make_scorer`, e.g. `{'R2': make_scorer(r2_score), ...}`.
2. In `evaluate`, compute `scores` with `cross_validate`, passing `scoring=scoring_function`.

If everything is right, you'll see results for each of the four combinations of encoding and log transform. **Which combination performs best? Why?**

In [ ]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_validate

def evaluate(X, y, scoring_function):
    for encoding in ['ordinal', 'onehot']:
        for xform in [False, True]:
            pipeline = ml_pipeline(encoder=encoding, log_scale=xform)
            scores = None  # WHAT GOES HERE???
            print(f"\nEncoding: {encoding}  Log Transform: {xform}\n****************************************")
            for key, values in scores.items():
                print(f"{key}: {values.mean():.3f} (+/- {values.std() * 2:.3f})")


scoring_function = None  # WHAT GOES HERE???

evaluate(X, y, scoring_function)

_Answer here_

# Section 2: Classifiers and cross-validation

In these exercises, you'll evaluate a logistic regression model on the iris dataset, first on a single train/test split and then with cross-validation.

**Iris has three classes**, so precision, recall, and F1 must be averaged across the classes. Pass `average='macro'` to `precision_score`, `recall_score`, and `f1_score` (the plain average of the per-class scores); in `cross_val_score`, the matching scorer name is `'f1_macro'`.

First, set up the data and pipeline:

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Load the iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Create a pipeline
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=42)
)

### **Exercise 3**: Evaluate on a Single Train-Test Split

Use `train_test_split` (with `test_size=0.2, random_state=42`) to create training and test sets, fit the pipeline on the training data, and then report on the test data:
- the confusion matrix (it will be 3×3: one row and column per species),
- precision, recall, and F1 (with `average='macro'`).

In [ ]:
# Your code here

### **Exercise 4**: Use cross_val_score for F1-score with Different Numbers of Folds

Use `cross_val_score` with `scoring='f1_macro'` to compute the F1-score using 3, 5, and 10 folds. Print the mean and standard deviation for each.

**Question:** Compare these results with your single split in Exercise 3. Which estimate of the model's performance would you trust more, and why?

In [ ]:
# Your code here
# Hint: Use cross_val_score with different cv values

_Answer here_